# 05b - Game Predictions (Binary Ensemble)

Predict full game statlines for starting pitchers using the binary model ensemble.

**Key differences from 05:**
- Uses 7 separate binary classifiers instead of one 7-class model
- Each outcome model has its own optimized features
- Should have better calibration for elite pitchers

In [9]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

from src.game_predictor_binary import GamePredictorBinary, format_prediction_summary
from src.data.mlb_api import get_games_with_lineups, check_lineup_status

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

## 1. Initialize Predictor

In [10]:
# Initialize predictor with binary ensemble
predictor = GamePredictorBinary(
    ensemble_dir='../models/binary_ensemble',
    preprocessor_path='../models/matchup_preprocessor.pkl',
    pitcher_profiles_path='../data/profiles/pitcher_arsenal.csv',
    batter_profiles_path='../data/profiles/batter_profiles.csv',
    pitcher_rolling_path='../data/profiles/pitcher_rolling.csv',
    batter_rolling_path='../data/profiles/batter_rolling.csv',
    park_factors_path= "../data/profiles/park_factors.csv"
)

print(f"Feature names: {len(predictor.feature_names)}")
print(f"Outcome classes: {predictor.outcome_classes}")

Feature names: 207
Outcome classes: ['1B', '2B', '3B', 'BB', 'HR', 'K', 'OUT']


## 2. Check Today's Games

In [11]:
# Check lineup availability
today = date.today().strftime("%Y-%m-%d")
print(f"Checking games for {today}...")

status = check_lineup_status(today)
print(f"\nTotal games: {status['total_games']}")
print(f"With lineups: {status['games_with_lineups']}")
print(f"With probable pitchers: {status['games_with_probable_pitchers']}")

print("\nGames:")
for game in status["games"]:
    away = game["away_team"]["abbreviation"]
    home = game["home_team"]["abbreviation"]
    away_p = game["away_pitcher"]["pitcher_name"] if game["away_pitcher"] else "TBD"
    home_p = game["home_pitcher"]["pitcher_name"] if game["home_pitcher"] else "TBD"
    lineups = "Lineups" if game["lineups_available"] else "No lineups"
    print(f"  {away} @ {home}: {away_p} vs {home_p} [{lineups}]")

Checking games for 2026-04-08...

Total games: 15
With lineups: 15
With probable pitchers: 15

Games:
  SD @ PIT: Michael King vs Mitch Keller [Lineups]
  KC @ CLE: Cole Ragans vs Joey Cantillo [Lineups]
  MIL @ BOS: Shane Drohan vs Sonny Gray [Lineups]
  BAL @ CWS: Kyle Bradish vs Sean Burke [Lineups]
  SEA @ TEX: Bryan Woo vs MacKenzie Gore [Lineups]
  LAD @ TOR: Shohei Ohtani vs Dylan Cease [Lineups]
  HOU @ COL: Cristian Javier vs Michael Lorenzen [Lineups]
  PHI @ SF: Aaron Nola vs Tyler Mahle [Lineups]
  STL @ WSH: Michael McGreevy vs Miles Mikolas [Lineups]
  ATL @ LAA: Grant Holmes vs Reid Detmers [Lineups]
  AZ @ NYM: Ryne Nelson vs David Peterson [Lineups]
  CHC @ TB: Colin Rea vs Joe Boyle [Lineups]
  CIN @ MIA: Brady Singer vs Eury Pérez [Lineups]
  ATH @ NYY: Luis Severino vs Will Warren [Lineups]
  DET @ MIN: Framber Valdez vs Bailey Ober [Lineups]


## 3. Run Predictions

In [12]:
# Predict all games for today
predictions = predictor.predict_day(today)

print(f"\n{'='*60}")
print(f"PREDICTIONS FOR {today}")
print(f"{'='*60}")

for game in predictions:
    print(f"\n{game['away_team']} @ {game['home_team']} ({game['status']})")
    print("-" * 40)
    
    if game["away_prediction"]:
        print(format_prediction_summary(game["away_prediction"]))
    
    if game["home_prediction"]:
        print(format_prediction_summary(game["home_prediction"]))
    
    if game["errors"]:
        print(f"  Errors: {game['errors']}")


PREDICTIONS FOR 2026-04-08

SD @ PIT (complete)
----------------------------------------
Michael King
  Target IP: 5.3, Sim IP: 5.3
  K: 6.5, BB: 1.7, H: 5.0, HR: 0.6
  Expected Runs: 1.96
  ERA (this start): 3.31
Mitch Keller
  Target IP: 4.8, Sim IP: 5.0
  K: 4.3, BB: 1.7, H: 4.9, HR: 0.6
  Expected Runs: 1.84
  ERA (this start): 3.31

KC @ CLE (complete)
----------------------------------------
Cole Ragans
  Target IP: 4.6, Sim IP: 4.7
  K: 5.1, BB: 1.8, H: 3.9, HR: 0.5
  Expected Runs: 1.44
  ERA (this start): 2.78
  Missing data for: Juan Brito
Joey Cantillo
  Target IP: 4.8, Sim IP: 4.7
  K: 4.8, BB: 1.8, H: 4.1, HR: 0.5
  Expected Runs: 1.76
  ERA (this start): 3.39

MIL @ BOS (complete)
----------------------------------------
Shane Drohan
  Target IP: 2.5, Sim IP: 2.7
  K: 2.6, BB: 1.1, H: 2.5, HR: 0.4
  Expected Runs: 1.24
  ERA (this start): 4.20
  Missing data for: Roman Anthony, Andruw Monasterio, Willson Contreras
Sonny Gray
  Target IP: 5.5, Sim IP: 5.7
  K: 6.0, BB: 1.

## 4. Detailed View - Single Game

In [13]:
# Show detailed predictions for first complete game
complete_games = [g for g in predictions if g['status'] == 'complete']

if complete_games:
    game = complete_games[0]
    print(f"Detailed view: {game['away_team']} @ {game['home_team']}")
    print("=" * 60)
    
    for side, pred in [("AWAY", game["away_prediction"]), ("HOME", game["home_prediction"])]:
        if pred:
            print(f"\n{side}: {pred['pitcher_name']}")
            print(f"Expected BF: {pred['expected_bf']}")
            print(f"\nExpected Stats:")
            for stat, val in pred['expected_stats'].items():
                print(f"  {stat}: {val}")
            
            print(f"\nPer-Batter Predictions (first time through):")
            for bp in pred['batter_predictions'][:9]:
                k_prob = bp['probabilities']['K']
                bb_prob = bp['probabilities']['BB']
                hr_prob = bp['probabilities']['HR']
                print(f"  {bp['batter_name'][:20]:20s}: K={k_prob:.1%}, BB={bb_prob:.1%}, HR={hr_prob:.1%}")
else:
    print("No complete games available")

Detailed view: SD @ PIT

AWAY: Michael King
Expected BF: 23.0

Expected Stats:
  K: 6.54
  BB: 1.7
  H: 4.98
  1B: 3.28
  2B: 1.05
  3B: 0.07
  HR: 0.58
  ER: 1.96
  IP_approx: 5.3
  ERA_approx: 3.31

Per-Batter Predictions (first time through):
  Oneil Cruz          : K=38.6%, BB=9.0%, HR=3.3%
  Brandon Lowe        : K=31.3%, BB=8.5%, HR=4.0%
  Bryan Reynolds      : K=28.6%, BB=10.7%, HR=3.2%
  Ryan O'Hearn        : K=25.0%, BB=9.3%, HR=2.5%
  Nick Yorke          : K=24.6%, BB=7.8%, HR=1.4%
  Nick Gonzales       : K=24.7%, BB=5.8%, HR=1.7%
  Spencer Horwitz     : K=20.9%, BB=10.8%, HR=2.1%
  Joey Bart           : K=35.1%, BB=9.3%, HR=2.5%
  Jake Mangum         : K=18.0%, BB=4.9%, HR=0.6%

HOME: Mitch Keller
Expected BF: 22.0

Expected Stats:
  K: 4.33
  BB: 1.7
  H: 4.92
  1B: 3.34
  2B: 0.92
  3B: 0.09
  HR: 0.57
  ER: 1.84
  IP_approx: 5.0
  ERA_approx: 3.31

Per-Batter Predictions (first time through):
  Ramón Laureano      : K=24.3%, BB=7.8%, HR=3.8%
  Fernando Tatis Jr.  : K=22.1

## 5. Compare with Yesterday (Completed Games)

In [14]:
# Get yesterday's games (should have lineups from boxscore)
yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Predictions for yesterday ({yesterday}):")

yesterday_preds = predictor.predict_day(yesterday)

for game in yesterday_preds:
    if game['status'] == 'complete':
        print(f"\n{game['away_team']} @ {game['home_team']}")
        
        if game["away_prediction"]:
            p = game["away_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")
        
        if game["home_prediction"]:
            p = game["home_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")

Predictions for yesterday (2026-04-07):

KC @ CLE
  Noah Cameron: 4.7 K, 2.4 BB, 5.9 H, ERA=3.99
  Gavin Williams: 5.4 K, 2.4 BB, 4.7 H, ERA=3.11

BAL @ CWS
  Trevor Rogers: 6.5 K, 2.2 BB, 6.1 H, ERA=3.86
  Shane Smith: 5.7 K, 2.2 BB, 4.2 H, ERA=3.61

AZ @ NYM
  Zac Gallen: 4.5 K, 1.8 BB, 5.8 H, ERA=4.79
  Freddy Peralta: 5.6 K, 2.3 BB, 4.7 H, ERA=2.93

SD @ PIT
  Nick Pivetta: 6.2 K, 1.7 BB, 5.0 H, ERA=3.45
  Paul Skenes: 5.9 K, 1.6 BB, 4.3 H, ERA=2.82

CHC @ TB
  Javier Assad: 2.9 K, 1.9 BB, 5.2 H, ERA=4.46
  Mason Englert: 2.5 K, 1.1 BB, 3.2 H, ERA=4.06

CIN @ MIA
  Andrew Abbott: 4.6 K, 1.9 BB, 5.8 H, ERA=4.51
  Sandy Alcantara: 7.0 K, 2.1 BB, 6.4 H, ERA=4.0

STL @ WSH
  Matthew Liberatore: 3.8 K, 1.3 BB, 4.8 H, ERA=3.32
  Cade Cavalli: 4.2 K, 1.7 BB, 4.4 H, ERA=3.97

MIL @ BOS
  Jacob Misiorowski: 6.8 K, 1.9 BB, 4.1 H, ERA=3.54
  Garrett Crochet: 7.3 K, 1.8 BB, 5.4 H, ERA=2.7

ATH @ NYY
  Aaron Civale: 4.5 K, 1.4 BB, 4.5 H, ERA=4.35
  Cam Schlittler: 6.0 K, 1.9 BB, 5.5 H, ERA=4.24

## 6. Elite Pitcher Check

Verify predictions for elite K pitchers are improved.

In [15]:
# Find elite pitchers in recent predictions
all_preds = yesterday_preds + predictions

elite_k_threshold = 7.0  # Expected Ks
elite_pitchers = []

for game in all_preds:
    for pred_key in ["away_prediction", "home_prediction"]:
        pred = game.get(pred_key)
        if pred and pred["expected_stats"]["K"] >= elite_k_threshold:
            elite_pitchers.append({
                "name": pred["pitcher_name"],
                "K": pred["expected_stats"]["K"],
                "BB": pred["expected_stats"]["BB"],
                "H": pred["expected_stats"]["H"],
                "ERA": pred["expected_stats"]["ERA_approx"],
                "BF": pred["expected_bf"],
            })

if elite_pitchers:
    print(f"Elite K predictions (>= {elite_k_threshold} K):")
    elite_df = pd.DataFrame(elite_pitchers).sort_values("K", ascending=False)
    print(elite_df.to_string(index=False))
else:
    print("No elite K predictions found in recent games")

Elite K predictions (>= 7.0 K):
           name    K   BB    H  ERA   BF
   Tarik Skubal 7.41 1.43 4.98 3.08 24.5
Garrett Crochet 7.34 1.80 5.39 2.70 25.0
Sandy Alcantara 7.01 2.08 6.44 4.00 27.0


## 7. Summary Table

In [16]:
# Create summary DataFrame
rows = []

for game in predictions:
    for side, pred_key in [("away", "away_prediction"), ("home", "home_prediction")]:
        pred = game.get(pred_key)
        if pred:
            s = pred["expected_stats"]
            rows.append({
                "Game": f"{game['away_team']}@{game['home_team']}",
                "Pitcher": pred["pitcher_name"],
                "BF": pred["expected_bf"],
                "K": s["K"],
                "BB": s["BB"],
                "H": s["H"],
                "HR": s["HR"],
                "ER": s["ER"],
                "IP": s["IP_approx"],
                "ERA": s["ERA_approx"],
            })

if rows:
    summary_df = pd.DataFrame(rows)
    print(f"\nPrediction Summary for {today}:")
    print(summary_df.to_string(index=False))
else:
    print("No predictions available")


Prediction Summary for 2026-04-08:
   Game          Pitcher   BF    K   BB    H   HR   ER  IP  ERA
 SD@PIT     Michael King 23.0 6.54 1.70 4.98 0.58 1.96 5.3 3.31
 SD@PIT     Mitch Keller 22.0 4.33 1.70 4.92 0.57 1.84 5.0 3.31
 KC@CLE      Cole Ragans 20.0 5.09 1.77 3.88 0.49 1.44 4.7 2.78
 KC@CLE    Joey Cantillo 21.0 4.80 1.82 4.13 0.53 1.76 4.7 3.39
MIL@BOS     Shane Drohan 15.0 2.64 1.08 2.49 0.39 1.24 2.7 4.20
MIL@BOS       Sonny Gray 24.0 6.02 1.92 5.14 0.48 1.98 5.7 3.15
BAL@CWS     Kyle Bradish 21.0 5.29 2.01 4.01 0.41 1.44 5.0 2.59
BAL@CWS       Sean Burke 20.0 4.25 1.99 3.61 0.74 1.93 4.0 4.34
SEA@TEX        Bryan Woo 23.0 6.05 1.80 5.43 0.75 2.23 5.7 3.54
SEA@TEX   MacKenzie Gore 21.5 5.62 2.36 4.45 0.57 2.07 5.0 3.73
LAD@TOR    Shohei Ohtani 19.5 5.38 1.48 4.07 0.46 1.43 4.7 2.75
LAD@TOR      Dylan Cease 22.0 6.21 2.03 4.12 0.85 2.13 4.7 4.11
HOU@COL  Cristian Javier 21.0 4.83 2.03 4.11 0.66 2.20 4.3 4.57
HOU@COL Michael Lorenzen 22.5 3.90 1.74 4.82 0.53 2.21 4.7 4.27
 PHI